[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/rubiks_cube/rubiks_cube.ipynb)

# Solving a Rubik's Cube

This example scrambles a Rubik's cube and solves it by bidirectional breadth-first search. The cube's 54 stickers move only by quarter turns of its six faces, and a random scramble of quarter turns is solved by searching outward from both the scrambled cube and the solved cube at once, advancing one quarter turn at a time from each end until the two frontiers share a state.

Each sticker is a point, and a quarter turn of a face is a rotor, `1 + plane` for the plane the face turns in. Its sandwich doubles lengths, so as a map on points halved, `((1 + plane) >> Point) * 0.5`, it turns by a quarter turn with coefficients 0 and ±1 only. Placed at integer coordinates, the stickers keep integer coordinates through any number of turns, so states compare without round-off. The twelve quarter-turn maps form one batch, and applying that batch to a batch of cube states turns every state every way at once, advancing an entire search frontier in one operation.

In Singmaster's notation the faces along $+x, -x, +y, -y, +z,$ and $-z$ read as $\mathrm{R}, \mathrm{L}, \mathrm{B}, \mathrm{F}, \mathrm{U},$ and $\mathrm{D}$ for clockwise quarter turns seen from outside and $\mathrm{R}', \mathrm{L}', \mathrm{B}', \mathrm{F}', \mathrm{U}',$ and $\mathrm{D}'$ for their inverses, generating the Rubik's cube group as a subgroup of the symmetric group $S_{48}$ on the 48 moving facelets.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from examples.util.animation import show_animation
from examples.geometry.rubiks_cube import render
from examples.geometry.rubiks_cube.core import codes, cubies, outward, solve, solved_cube, squares

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
# Stickers, cubies, and corners are points (antivectors), and faces are planes through the origin (vectors).
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Turn = ga.gatype((Point, Point))

# Each of the six faces is a plane along an axis on either side; `face & point` reads how far a point reaches along it.
sides = np.array([1, -1])
faces = sides[:, None] * mv.basis()                                         # [sides, axes] Plane+
# Each face turns in the bivector plane perpendicular to its axis, oriented clockwise seen from outside as in Singmaster's notation.
planes = sides[:, None] * stack([mv.yz, mv.zx, mv.xy])                      # [sides, axes] Bivector+
# Each quarter turn is a map on points, clockwise and anticlockwise around each face.
directions = np.array([1, -1])
turns = ((1 + directions[:, None, None] * planes) >> Point) * 0.5           # [directions, sides, axes] Point- <- Point-

## 1. Quarter turns as maps

A face is a plane, a side times an axis, and a sticker belongs to a face's turning layer when `face & stickers > 1`. Each quarter-turn map turns the stickers in that layer and leaves the remaining stickers in place, with coefficients 0 and ±1 only.

In [ ]:
def turn_layer(stickers: Point, turn: Turn, face: Plane) -> Point:
    """The stickers after a quarter turn of `face` turns the stickers in its outer layer and leaves the rest in place."""
    layer = (face & stickers) > 1                                           # [...]
    return stickers + layer * (turn(stickers) - stickers)                   # [...] Point-


solved_stickers, solved_corners, colours = solved_cube()                    # [stickers] Point-, [stickers, corners] Point-, [stickers]

In [ ]:
print("a quarter turn of the face along +z, a map on points:\n", turns[0, 0, 2].coefficient_table())

## 2. Scrambling and solving

Ten random quarter turns, none undoing the one before, scramble the cube. Bidirectional breadth-first search widens the states reachable from the scrambled cube and from the solved cube alternately, one quarter turn at a time, applying all twelve quarter turns to every state of the active frontier at once and keeping only states not visited before; where the two frontiers meet, the turns from the scramble to the meeting state and back from there to the solved cube solve the scramble.

In [ ]:
# The scramble's quarter-turn count and random seed, and the search depth from each end.
n_scramble, seed, depth = 10, 4, 5

In [ ]:
def random_turns(rng: np.random.Generator) -> Iterator[np.ndarray]:
    """An endless stream of random quarter turns `(direction, side, axis)` with no move immediately undoing the one before."""
    previous_move = rng.integers((2, 2, 3))
    yield previous_move
    while True:
        move = rng.integers((2, 2, 3))
        if (1 - move[0], *move[1:]) != tuple(previous_move):
            previous_move = move
            yield move


def scramble(stickers: Point, moves: np.ndarray) -> Point:
    """The stickers after applying the sequence of quarter turns `moves` one after another."""
    for direction, side, axis in moves:
        stickers = turn_layer(stickers, turns[direction, side, axis], faces[side, axis])  # [stickers] Point-
    return stickers


moves = np.array(list(islice(random_turns(np.random.default_rng(seed)), n_scramble)))  # [turns, 3]
scrambled_stickers = scramble(solved_stickers, moves)                       # [stickers] Point-
solution = solve(scrambled_stickers, solved_stickers, depth)                # [turns, 3]

In [ ]:
print("scramble, direction side axis:\n", moves)
print("solution:\n", solution)

## 3. Animating the scramble and solution

The animation turns each layer smoothly by the rotor of a growing fraction of the quarter turn, `(plane * (-angle / 2)).exp()`, exposing the dark inner cut between the turning layer and the rest of the cube through the scramble and then the solution back to the solved state.

In [ ]:
# The frames per quarter turn, and each frame's duration in milliseconds.
n_steps, frame_duration = 9, 40

In [ ]:
def animate(stickers: Point, corners: Point, moves: np.ndarray, n_steps: int) -> Iterator[tuple[Point, Point]]:
    """The sticker corners and the two sides of the inner cut across `moves`, with each quarter turn interpolated smoothly over `n_steps` frames."""
    for direction, side, axis in moves:
        layer = (faces[side, axis] & stickers) > 1                          # [stickers]
        # The cut consists of the inner square faces of the turning layer's cubies.
        cut = 2 * cubies[outward[:, side, axis]][:, None] + squares[1 - side, axis]  # [layer_cubies, corners] Point-
        for fraction in np.arange(n_steps) / n_steps:
            rotor = (planes[side, axis] * (np.pi / 4 * fraction * directions[direction])).exp()  # [] Rotor+
            yield corners + layer[:, None] * ((rotor >> corners) - corners), stack([cut, rotor >> cut])
        turn, face = turns[direction, side, axis], faces[side, axis]
        stickers, corners = turn_layer(stickers, turn, face), corners + layer[:, None] * (turn(corners) - corners)
    yield corners, stack([cut, cut])


states = animate(solved_stickers, solved_corners, np.concatenate([moves, solution]), n_steps)
show_animation(render.animate(states, colours), "rubiks_cube", frame_duration)

In [ ]:
# checks
# Each quarter-turn map has coefficients 0 and ±1 only.
assert np.isin(turns.kernel, [-1, 0, 1]).all()
# The solution restores the scrambled cube to the solved state in no more turns than the scramble.
assert (codes(scramble(scrambled_stickers, solution)[None]) == codes(solved_stickers[None])).all()
assert len(solution) <= n_scramble